# Ongoing CV ISI Table

This notebook produces a table of the CV and mean firing rate statistics for the ongoing data.

In [ ]:
# Autoreloading of modules and files without restarting notebook.
%reload_ext autoreload
%autoreload 2

import neo
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from rvm_analysis.utils import import_spontaneous_cells
from rvm_analysis.data_loaders import read_dataset_by_protocol
from rvm_analysis.utils import import_neutral_cells_extra, import_neutral_cells
from elephant.statistics import cv, isi

#! Force the backend back to matplotlib inline (after importing viziphant).
%matplotlib inline
print(plt.get_backend())

module://matplotlib_inline.backend_inline


In [15]:
table_save_path = Path("ongoing_cell_summary_table.tex")

In [ ]:
def deprecated_data_loading_from_spike2():
    dm_spontaneous = import_spontaneous_cells()
    dm_neutral = import_neutral_cells()
    dm_neutral_extra = import_neutral_cells_extra()

dm_combined = read_dataset_by_protocol(['ongoing','evoked/ongoing'])

<sonpy.SonFile> object owning file ..\..\data\Spontaneous_Recordings_Melissa_Zhigang\Pared_Spike_Files\CA_ongoing01a_cut_pared.smrx
    DataType.AdcMark: 1, Spikes
    DataType.EventFall: 4, EKG
    DataType.EventRise: 6, heat
    DataType.EventRise: 7, flick
    DataType.Marker: 30, Keyboard
<sonpy.SonFile> object owning file ..\..\data\Spontaneous_Recordings_Melissa_Zhigang\Pared_Spike_Files\CA_ongoing01b_cut_pared.smrx
    DataType.AdcMark: 1, nw-1
    DataType.EventFall: 4, EKG
    DataType.EventRise: 6, heat
    DataType.EventRise: 7, flick
    DataType.Marker: 30, Keyboard
<sonpy.SonFile> object owning file ..\..\data\Spontaneous_Recordings_Melissa_Zhigang\Pared_Spike_Files\CA_ongoing02a_cut_pared.smrx
    DataType.AdcMark: 1, nw-1
    DataType.EventFall: 4, EKG
    DataType.EventRise: 6, heat
    DataType.EventRise: 7, flick
    DataType.Marker: 30, Keyboard
<sonpy.SonFile> object owning file ..\..\data\Spontaneous_Recordings_Melissa_Zhigang\Pared_Spike_Files\CA_ongoing02b_cut_p

In [ ]:
statistics_list = []

cell: neo.SpikeTrain
for cell in dm_combined.spiketrain_iterator(cell_type='All'):
    # The SpikeTrain type guarantees that isi(cell) has a magnitude, and cell has a file origin
    statistics_list.append([cell.file_origin.split("\\")[-1][:-5],cell.name,cv(isi(cell)),np.mean(isi(cell).magnitude)]) #type: ignore

cell_statistics_table = pd.DataFrame(statistics_list, columns=["Experiment","Cell Type","CV","ISI (s)"])

In [18]:
cell_statistics_table.sort_values(by='Cell Type',ascending=False)

,Experiment,Cell Type,CV,ISI (s)
0,CA_ongoing01a_cut_pared,ON,3.281597,0.129698
16,CA_ongoing05a_cut_pared,ON,1.947593,0.046141
31,ZS Naive 4-18-24 #1 ON&OFF@2120_cut_pared,ON,6.871174,0.103535
30,ZS Naive 4-17-24 a ON@1695_cut_pared,ON,9.956630,0.125085
29,Naive 4-16-24 c OFF&ON@2935_cut_pared,ON,3.639587,0.569416
...,...,...,...,...
53,HDB_Bic125_cut_truncated,Neutral,0.300697,0.115213
52,HDB_Bic125_cut_truncated,Neutral,0.166447,0.086106
51,HDB_Bic115_cut_truncated,Neutral,0.215445,0.036625
50,HDB_Bic09_cut_truncated,Neutral,0.110625,0.089078


Now we make a table which has the width of the cv, isi for each cell.

In [22]:
grouped = cell_statistics_table.groupby("Cell Type")

summary_with_std = grouped.agg({
    "CV": ["mean", "std"],
    "ISI (s)": ["mean", "std"]
}).reset_index()

# Flatten column names
summary_with_std.columns = ["Cell Type", "CV_mean", "CV_std", "ISI_mean", "ISI_std"]

In [23]:
summary_with_std

,Cell Type,CV_mean,CV_std,ISI_mean,ISI_std
0,Neutral,0.347861,0.263788,0.165018,0.250237
1,OFF,3.698732,3.086583,0.835044,1.356912
2,ON,4.908843,4.319423,1.248436,4.799301


In [24]:
summary_with_std["CV (mean ± std)"] = summary_with_std.apply(
    lambda row: f"{row['CV_mean']:.3f} ± {row['CV_std']:.3f}", axis=1)

summary_with_std["ISI (s) (mean ± std)"] = summary_with_std.apply(
    lambda row: f"{row['ISI_mean']:.3f} ± {row['ISI_std']:.3f}", axis=1)

latex_ready = summary_with_std[["Cell Type", "CV (mean ± std)", "ISI (s) (mean ± std)"]]

latex_ready.to_latex(table_save_path, index=False, escape=False)